### Links and Resources
| Concept / Command | Description |
|---|---|
| [Window operations](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time) | Event-time windowing in Structured Streaming |
| [`window()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.window.html) | Tumbling and sliding windows |
| [`session_window()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.session_window.html) | Gap-based session windows |
| [Watermarks](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#handling-late-data-and-watermarking) | Handling late data and bounding state |
| [`dropDuplicates`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#streaming-deduplication) | Streaming deduplication with watermark |

In [6]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Windowing and Watermarks

Event-time windows and watermarks are the tools that make **stateful streaming aggregations** both correct and bounded in memory.

Without them, a `groupBy` over a stream would either grow state forever (memory exhaustion) or refuse to emit results in `append` mode. This notebook shows both problems and their fix.

Topics:
1. Event time vs processing time
2. Tumbling windows — non-overlapping fixed-size buckets
3. Sliding windows — overlapping fixed-size buckets
4. Session windows — gap-based dynamic buckets
5. Why watermarks are required
6. How watermarks work
7. Late data — what is accepted vs dropped
8. `dropDuplicates` with a watermark
9. Output modes for windowed queries

## 1. Event time vs processing time

Every event has two timestamps:

| Timestamp | Meaning | Where it lives |
|---|---|---|
| **Event time** | When the event *happened* — set by the producer | A column in your data (e.g. `order_placed_at`) |
| **Processing time** | When Spark *sees* the event | `current_timestamp()` at ingestion |

For analytics — "how many orders per minute?" — **event time** is almost always what you mean.
Processing time shifts with network delays, retries, and micro-batch scheduling, giving unstable window boundaries.

The rate source generates a `timestamp` column that is processing time only. For this notebook we treat it as event time (no delivery delay in the demo).

In [7]:
import time, uuid
from pyspark.sql import functions as F

# rate-micro-batch (not plain rate) so availableNow can terminate the stream — see notebook 02.
events = (spark.readStream
    .format("rate-micro-batch")
    .option("rowsPerBatch", 40)
    .option("numPartitions", 1)
    .load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumnRenamed("value",     "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 100 + 1).cast("int")))

events.printSchema()
print("isStreaming:", events.isStreaming)

root
 |-- event_time: timestamp (nullable = true)
 |-- event_id: long (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: integer (nullable = true)

isStreaming: True


### Helper — peek at a streaming DataFrame

Windowed aggregations require `"complete"` or `"update"` output mode (not `"append"`).
The helper accepts an `output_mode` parameter.

In [8]:
def peek(stream_df, name, seconds=None, limit=25, output_mode="complete", order_by=None):
    """Run a streaming DF into the memory sink with availableNow + uuid-suffixed checkpoint, await completion, return rows.
    Serverless rejects continuous triggers and implicit temp checkpoints — see notebook 02.
    `seconds` is kept for back-compat with earlier call sites; ignored under availableNow."""
    checkpoint = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/{name}_{uuid.uuid4().hex[:8]}"
    q = (stream_df.writeStream
         .format("memory")
         .queryName(name)
         .outputMode(output_mode)
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .start())
    q.awaitTermination()
    order_clause = f"ORDER BY {order_by}" if order_by else ""
    return spark.sql(f"SELECT * FROM {name} {order_clause} LIMIT {limit}")

## 2. Tumbling windows

A **tumbling window** is a fixed-size, non-overlapping bucket. Every event belongs to exactly one window.

```
  ──────────────────────────────────────────────────────── event_time ──▶
  │       10 s        │       10 s        │       10 s        │
  │    window 1       │    window 2       │    window 3       │
  └───────────────────┴───────────────────┴───────────────────┘
```

`window(col("event_time"), "10 seconds")` returns a `{start, end}` struct that you group by.

In [9]:
tumbling = (events
    .groupBy(
        F.window("event_time", "10 seconds"),
        "user_id"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "user_id", "event_count", "total_amount",
    ))

display(peek(tumbling, "tumbling_demo", seconds=12, order_by="window_start, user_id"))

+-------------------+-------------------+-------+-----------+------------+
|window_start       |window_end         |user_id|event_count|total_amount|
+-------------------+-------------------+-------+-----------+------------+
|1970-01-01 00:00:00|1970-01-01 00:00:10|0      |8          |421         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|1      |8          |441         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|2      |8          |430         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|3      |8          |359         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|4      |8          |352         |
+-------------------+-------------------+-------+-----------+------------+



## 3. Sliding windows

A **sliding window** has a fixed size but advances by a *slide interval* smaller than the window size.
One event can appear in **multiple** windows — whichever windows contain its timestamp.

```
  ──────────────────────────────────────────────────────── event_time ──▶
  │       20 s window        │
            │       20 s window        │
                      │       20 s window        │
                 slide = 10 s    slide = 10 s
```

`window(col("event_time"), "20 seconds", "10 seconds")` — size 20 s, slide every 10 s.

In [10]:
sliding = (events
    .groupBy(
        F.window("event_time", "20 seconds", "10 seconds"),
        "user_id"
    )
    .agg(F.count("*").alias("event_count"))
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "user_id", "event_count",
    ))

display(peek(sliding, "sliding_demo", seconds=12, order_by="window_start, user_id"))

+-------------------+-------------------+-------+-----------+
|window_start       |window_end         |user_id|event_count|
+-------------------+-------------------+-------+-----------+
|1969-12-31 23:59:50|1970-01-01 00:00:10|0      |8          |
|1969-12-31 23:59:50|1970-01-01 00:00:10|1      |8          |
|1969-12-31 23:59:50|1970-01-01 00:00:10|2      |8          |
|1969-12-31 23:59:50|1970-01-01 00:00:10|3      |8          |
|1969-12-31 23:59:50|1970-01-01 00:00:10|4      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:20|0      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:20|1      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:20|2      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:20|3      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:20|4      |8          |
+-------------------+-------------------+-------+-----------+



## 4. Session windows

A **session window** groups events by *inactivity gap*. If no events arrive for longer than the gap, the window closes; the next event starts a new session.

Sessions are **dynamic** — their length varies by user activity, not by a fixed clock.

```
  user 0:  ──●●●──────────────────●●────●●●──
             session 1            s2    s3
             (gap > 5 s)               (gap > 5 s)
```

`session_window(col("event_time"), "5 seconds")` — gap threshold 5 s.

In [11]:
session = (events
    .groupBy(
        F.session_window("event_time", "5 seconds"),
        "user_id"
    )
    .agg(F.count("*").alias("event_count"))
    .select(
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "user_id", "event_count"))

display(peek(session, "session_demo", seconds=12, order_by="session_start, user_id"))

+-------------------+-------------------+-------+-----------+
|session_start      |session_end        |user_id|event_count|
+-------------------+-------------------+-------+-----------+
|1970-01-01 00:00:00|1970-01-01 00:00:05|0      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:05|1      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:05|2      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:05|3      |8          |
|1970-01-01 00:00:00|1970-01-01 00:00:05|4      |8          |
+-------------------+-------------------+-------+-----------+



## 5. Why watermarks are required

Without a watermark, Spark must keep state for **every window ever opened** — because a late event could arrive at any time and update an old window.

On a stream running for days, this means state for thousands of 10-second windows, growing until the executor runs out of memory.

A second problem: in `append` output mode, Spark emits a window's result only when it is certain the window is **final** (no more late data). Without a watermark, Spark can never be certain — nothing is ever emitted in `append` mode.

**A watermark is the contract you make with Spark:** "events older than X seconds behind the latest event I have seen will never arrive. You can drop their state."

In [12]:
# Without watermark — state grows forever
no_wm = (events
    .groupBy(F.window("event_time", "10 seconds"))
    .agg(F.count("*").alias("event_count")))

print("Without watermark: state for every window retained forever.")
print("On a long-running stream this exhausts executor memory.\n")

# With watermark — add withWatermark BEFORE groupBy
with_wm = (events
    .withWatermark("event_time", "10 seconds")    # <── add this
    .groupBy(F.window("event_time", "10 seconds"))
    .agg(F.count("*").alias("event_count")))

print("With watermark: windows whose end < (max_event_time − 10 s) are evicted from state.")

Without watermark: state for every window retained forever.
On a long-running stream this exhausts executor memory.

With watermark: windows whose end < (max_event_time − 10 s) are evicted from state.


## 6. How watermarks work

```
  watermark threshold        = 10 seconds
  max event_time seen so far = 12:00:30
  current watermark          = 12:00:20   ← max_event_time − threshold

  event arrives: event_time = 12:00:05
    → 12:00:05 < 12:00:20  → LATE  — dropped, window state already evicted

  event arrives: event_time = 12:00:25
    → 12:00:25 ≥ 12:00:20  → ON TIME — processed normally
```

**Rule:** an event is accepted if `event_time ≥ max_seen_event_time − threshold`.

**What the watermark controls:**
- **State eviction** — windows that ended before the watermark are dropped from the state store.
- **`append` output** — a window result is emitted only once the watermark passes its end.

Setting threshold too small → more late data dropped, less state, faster output.
Setting threshold too large → more late-data tolerance, more memory, output delayed.

In [13]:
watermarked = (events
    .withWatermark("event_time", "10 seconds")
    .groupBy(
        F.window("event_time", "10 seconds"),
        "user_id"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "user_id", "event_count", "total_amount",
    ))

display(peek(watermarked, "watermarked_demo", seconds=15, order_by="window_start, user_id"))

+-------------------+-------------------+-------+-----------+------------+
|window_start       |window_end         |user_id|event_count|total_amount|
+-------------------+-------------------+-------+-----------+------------+
|1970-01-01 00:00:00|1970-01-01 00:00:10|0      |8          |332         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|1      |8          |445         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|2      |8          |372         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|3      |8          |306         |
|1970-01-01 00:00:00|1970-01-01 00:00:10|4      |8          |522         |
+-------------------+-------------------+-------+-----------+------------+



## 7. Late data — what is accepted vs dropped

With a watermark of 15 seconds, two parallel streams demonstrate late data handling:
- `on_time` stream: `event_time = now` — all accepted
- `late_30s` stream: `event_time = now − 30 s` — all dropped (30 s > 15 s threshold)

Late events are **silently dropped** — no error raised. If late data is business-critical, capture it in a dead-letter Delta table using `foreachBatch`.

In [ ]:
on_time = (spark.readStream.format("rate-micro-batch").option("rowsPerBatch", 8).load()
    .withColumn("event_time", F.col("timestamp"))
    .withColumn("stream",     F.lit("on_time"))
    .withColumnRenamed("value", "event_id"))

late = (spark.readStream.format("rate-micro-batch").option("rowsPerBatch", 8).load()
    .withColumn("event_time", F.col("timestamp") - F.expr("INTERVAL 30 SECONDS"))
    .withColumn("stream",     F.lit("late_30s"))
    .withColumnRenamed("value", "event_id"))

combined = on_time.union(late)

windowed = (combined
    .withWatermark("event_time", "15 seconds")
    .groupBy(F.window("event_time", "10 seconds"), "stream")
    .agg(F.count("*").alias("events")))

checkpoint = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/late_demo_{uuid.uuid4().hex[:8]}"
q = (windowed.writeStream
     .format("memory").queryName("late_demo")
     .outputMode("complete")
     .option("checkpointLocation", checkpoint)
     .trigger(availableNow=True)
     .start())

q.awaitTermination()

print("on_time windows are populated; late_30s windows fall behind the 15s watermark and are dropped:")
spark.sql(
    "SELECT window.start AS window_start, window.end AS window_end, stream, events "
    "FROM late_demo ORDER BY window_start, stream"
).show(truncate=False)

## 8. `dropDuplicates` with a watermark

`dropDuplicates` on a stream keeps a hash set of every ID it has ever seen — unbounded state without a watermark.

With `.withWatermark("event_time", "30 seconds").dropDuplicates(["event_id"])`:
- IDs with `event_time` older than 30 s behind the watermark are evicted from state.
- Duplicates arriving more than 30 s late are **not** caught — they pass through.

**Important:** `withWatermark` must precede `dropDuplicates` in the call chain.

In [ ]:
# value // 2 maps two consecutive events to the same dedup_id → deliberate duplicates
events_with_dupes = (spark.readStream
    .format("rate-micro-batch").option("rowsPerBatch", 24).load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumn("dedup_id", (F.col("value") // 2).cast("long"))
    .withColumn("user_id",  (F.col("value") % 5).cast("int")))

deduped = (events_with_dupes
    .withWatermark("event_time", "30 seconds")   # must precede dropDuplicates
    .dropDuplicates(["dedup_id"]))

cp_raw   = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/raw_dupes_{uuid.uuid4().hex[:8]}"
cp_dedup = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/deduped_{uuid.uuid4().hex[:8]}"

q_raw = (events_with_dupes.writeStream.format("memory").queryName("raw_dupes")
         .outputMode("append").option("checkpointLocation", cp_raw)
         .trigger(availableNow=True).start())
q_dedup = (deduped.writeStream.format("memory").queryName("deduped")
           .outputMode("append").option("checkpointLocation", cp_dedup)
           .trigger(availableNow=True).start())

q_raw.awaitTermination()
q_dedup.awaitTermination()

raw   = spark.sql("SELECT COUNT(*), COUNT(DISTINCT dedup_id) FROM raw_dupes").collect()[0]
dedup = spark.sql("SELECT COUNT(*), COUNT(DISTINCT dedup_id) FROM deduped").collect()[0]

print(f"Raw stream:    {raw[0]:>4} rows,  {raw[1]:>4} distinct dedup_ids")
print(f"Deduped stream:{dedup[0]:>4} rows,  {dedup[1]:>4} distinct dedup_ids")
print("Deduped row count ≈ distinct IDs from raw (each consecutive pair collapsed to one row)")

## 9. Output modes for windowed queries

| Output mode | What is emitted each batch | State evicted? | Requires watermark? |
|---|---|---|---|
| `complete` | The **entire** result table — every window | No — state kept forever | No (but state grows) |
| `update` | Only windows that **changed** this batch | No — state kept forever | No (but state grows) |
| `append` | Windows that are **finalised** (watermark passed their end) | **Yes** — old windows dropped | **Yes** |

For production:
- **`append`** with watermark → final, immutable rows. Use with Delta or Kafka.
- **`complete`** → only practical for small, bounded key spaces (e.g. per-user counters with few users).
- **`update`** → useful with `foreachBatch` + Delta `MERGE` for low-latency upsert patterns.

In [ ]:
cp_complete = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/out_complete_{uuid.uuid4().hex[:8]}"
q_complete = (events
    .withWatermark("event_time", "10 seconds")
    .groupBy("user_id")
    .agg(F.count("*").alias("events"), F.sum("amount").alias("total"))
    .writeStream.format("memory").queryName("out_complete")
    .outputMode("complete")
    .option("checkpointLocation", cp_complete)
    .trigger(availableNow=True)
    .start())

q_complete.awaitTermination()

print("complete mode — all user_ids, totals accumulated across all received batches:")
spark.sql("SELECT user_id, events, total FROM out_complete ORDER BY user_id").show()

cp_update = f"/Volumes/population_metrics/landing/datasets/streaming_demo/_checkpoints/out_update_{uuid.uuid4().hex[:8]}"
q_update = (events
    .withWatermark("event_time", "10 seconds")
    .groupBy("user_id")
    .agg(F.count("*").alias("events"), F.sum("amount").alias("total"))
    .writeStream.format("memory").queryName("out_update")
    .outputMode("update")
    .option("checkpointLocation", cp_update)
    .trigger(availableNow=True)
    .start())

q_update.awaitTermination()

print("\nupdate mode — memory sink accumulates updates; final snapshot looks the same:")
spark.sql("SELECT user_id, events, total FROM out_update ORDER BY user_id").show()

## Takeaways

- Use **event time** (a column in your data) for windowing — stable regardless of delivery delays.
- **Tumbling windows** (`window("ts", "10 s")`) — one event, one window; non-overlapping.
- **Sliding windows** (`window("ts", "20 s", "5 s")`) — one event, multiple windows; overlapping.
- **Session windows** (`session_window("ts", "5 s")`) — gap-based; natural for user activity.
- **Always add `.withWatermark(column, delay)`** before any stateful operator in production. Without it, state grows unboundedly.
- The watermark is `max_seen_event_time − threshold`. Events below it are dropped silently.
- **`dropDuplicates`** requires a watermark; `withWatermark` must precede it in the chain.
- **`append`** output mode emits final, immutable windows — requires a watermark.
- **`complete`** re-emits the whole table each batch — only practical for small key spaces.